In [2]:
pip install scikeras

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
# =========================
# INSTALL (run once if needed)
# =========================
# pip install kagglehub[pandas-datasets] scikeras tensorflow scikit-learn pandas matplotlib

# =========================
# IMPORTS
# =========================
import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from scikeras.wrappers import KerasClassifier

# =========================
# LOAD DATASET
# =========================
file_path = "Churn_Modelling.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "shrutimechlearn/churn-modelling",
    file_path
)

print("Dataset Loaded")
print(df.head())

# =========================
# PREPROCESSING
# =========================
df = df.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

# Encode Gender
le = LabelEncoder()
df["Gender"] = le.fit_transform(df["Gender"])

# One-hot encode Geography
df = pd.get_dummies(df, columns=["Geography"], drop_first=True)

# Features & target
X = df.drop("Exited", axis=1)
y = df["Exited"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =========================
# BUILD MODEL FUNCTION
# =========================
def build_model(neurons=32, dropout_rate=0.3, optimizer='adam'):
    model = keras.Sequential()

    model.add(layers.Dense(neurons, activation='relu', input_shape=(X_train.shape[1],)))
    model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(neurons, activation='relu'))
    model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

# =========================
# SCIKERAS WRAPPER
# =========================
model = KerasClassifier(
    model=build_model,
    verbose=0
)

# =========================
# GRID SEARCH PARAMETERS
# =========================
param_grid = {
    'model__neurons': [16, 32],
    'model__dropout_rate': [0.2, 0.3],
    'optimizer': ['adam', 'rmsprop'],
    'batch_size': [16, 32],
    'epochs': [50]
}

# =========================
# CROSS VALIDATION
# =========================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    n_jobs=-1
)

print("Running Grid Search...")
grid_result = grid.fit(X_train, y_train)

print("\nBest Parameters:", grid_result.best_params_)
print("Best Cross-Validation Accuracy:", grid_result.best_score_)

# =========================
# MODEL CHECKPOINT
# =========================
checkpoint = keras.callbacks.ModelCheckpoint(
    "best_churn_model.h5",
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

# =========================
# TRAIN FINAL MODEL
# =========================
best_params = grid_result.best_params_

final_model = build_model(
    neurons=best_params['model__neurons'],
    dropout_rate=best_params['model__dropout_rate'],
    optimizer=best_params['optimizer']
)

history = final_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=best_params['epochs'],
    batch_size=best_params['batch_size'],
    callbacks=[checkpoint],
    verbose=1
)

# =========================
# EVALUATION
# =========================
loss, accuracy = final_model.evaluate(X_test, y_test)

print("\nFinal Test Accuracy:", accuracy)

# =========================
# OPTIONAL: LOAD BEST MODEL
# =========================
best_model = keras.models.load_model("best_churn_model.h5")
best_loss, best_acc = best_model.evaluate(X_test, y_test)

print("Best Saved Model Accuracy:", best_acc)

/tmp/ipykernel_2139749/2707093574.py:28: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Dataset Loaded
   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63       0 

2026-04-17 13:51:38.832443: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-17 13:51:38.860857: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-17 13:51:38.875758: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-17 13:51:38.875780: I


Best Parameters: {'batch_size': 16, 'epochs': 50, 'model__dropout_rate': 0.2, 'model__neurons': 16, 'optimizer': 'adam'}
Best Cross-Validation Accuracy: 0.861
Epoch 1/50


/opt/intel/oneapi/intelpython/lib/python3.11/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8453 - loss: 0.3723 - val_accuracy: 0.8562 - val_loss: 0.3530
Epoch 21/50

Epoch 21: val_accuracy did not improve from 0.85625
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8475 - loss: 0.3732 - val_accuracy: 0.8556 - val_loss: 0.3562
Epoch 22/50

Epoch 22: val_accuracy did not improve from 0.85625
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8487 - loss: 0.3690 - val_accuracy: 0.8544 - val_loss: 0.3542
Epoch 23/50

Epoch 23: val_accuracy did not improve from 0.85625
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8469 - loss: 0.3743 - val_accuracy: 0.8537 - val_loss: 0.3528
Epoch 24/50

Epoch 24: val_accuracy improved from 0.85625 to 0.85688, saving model to best_churn_model.h5
- loss: 0.3642  

400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8491 - loss: 0.3720 - val_accuracy: 0.8569 - val_loss: 0.3527
Epoch 25/50

Epoch 25: val_accuracy did not improve from 0.85688
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8467 - loss: 0.3649 - val_accuracy: 0.8556 - val_loss: 0.3486
Epoch 26/50

Epoch 26: val_accuracy did not improve from 0.85688
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8436 - loss: 0.3702 - val_accuracy: 0.8562 - val_loss: 0.3499
Epoch 27/50

Epoch 27: val_accuracy did not improve from 0.85688
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8491 - loss: 0.3647 - val_accuracy: 0.8569 - val_loss: 0.3474
Epoch 28/50

Epoch 28: val_accuracy improved from 0.85688 to 0.85938, saving model to best_churn_model.h5
- loss: 0.3608    

400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8477 - loss: 0.3614 - val_accuracy: 0.8594 - val_loss: 0.3464
Epoch 29/50

Epoch 29: val_accuracy did not improve from 0.85938
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8498 - loss: 0.3678 - val_accuracy: 0.8587 - val_loss: 0.3456
Epoch 30/50

Epoch 30: val_accuracy improved from 0.85938 to 0.86250, saving model to best_churn_model.h5
- loss: 0.3607    

400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8491 - loss: 0.3690 - val_accuracy: 0.8625 - val_loss: 0.3506
Epoch 31/50

Epoch 31: val_accuracy did not improve from 0.86250
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8544 - loss: 0.3630 - val_accuracy: 0.8569 - val_loss: 0.3457
Epoch 32/50

Epoch 32: val_accuracy did not improve from 0.86250
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8542 - loss: 0.3569 - val_accuracy: 0.8569 - val_loss: 0.3435
Epoch 33/50

Epoch 33: val_accuracy did not improve from 0.86250
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8528 - loss: 0.3586 - val_accuracy: 0.8594 - val_loss: 0.3452
Epoch 34/50

Epoch 34: val_accuracy did not improve from 0.86250
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8552 - loss: 0.3498 - val_accuracy: 0.8575 - val_loss: 0.3410
Epoch 35/50

Epoch 35: val_accuracy did not improve from 0.86250
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8528 - loss: 0.3580 - val_accuracy:


Final Test Accuracy: 0.8575000166893005
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8580 - loss: 0.3469   
Best Saved Model Accuracy: 0.8579999804496765


In [6]:
# =========================
# INSTALL (run once if needed)
# =========================
# pip install kagglehub[pandas-datasets] scikeras tensorflow scikit-learn pandas matplotlib

# =========================
# IMPORTS
# =========================
import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from scikeras.wrappers import KerasClassifier

# =========================
# LOAD DATASET
# =========================
file_path = "Churn_Modelling.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "shrutimechlearn/churn-modelling",
    file_path
)

print("Dataset Loaded")
print(df.head())

# =========================
# PREPROCESSING
# =========================
df = df.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

# Encode Gender
le = LabelEncoder()
df["Gender"] = le.fit_transform(df["Gender"])

# One-hot encode Geography
df = pd.get_dummies(df, columns=["Geography"], drop_first=True)

# Features & target
X = df.drop("Exited", axis=1)
y = df["Exited"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =========================
# BUILD MODEL FUNCTION
# =========================
def build_model(neurons=64, dropout_rate=0.2, optimizer='adam'):
    model = keras.Sequential()

    # Layer 1
    model.add(layers.Dense(neurons, activation='relu', input_shape=(X_train.shape[1],)))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(dropout_rate))

    # Layer 2
    model.add(layers.Dense(neurons//2, activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(dropout_rate))

    # Layer 3
    model.add(layers.Dense(neurons//4, activation='relu'))

    # Output Layer
    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

# =========================
# SCIKERAS WRAPPER
# =========================
model = KerasClassifier(
    model=build_model,
    verbose=0
)

# =========================
# GRID SEARCH PARAMETERS
# =========================
param_grid = {
    'model__neurons': [64, 128],
    'model__dropout_rate': [0.1, 0.2],
    'optimizer': ['adam'],
    'batch_size': [32],
    'epochs': [50]
}

# =========================
# CROSS VALIDATION
# =========================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    n_jobs=-1
)

print("Running Grid Search...")
grid_result = grid.fit(X_train, y_train)

print("\nBest Parameters:", grid_result.best_params_)
print("Best Cross-Validation Accuracy:", grid_result.best_score_)

# =========================
# CALLBACKS
# =========================
checkpoint = keras.callbacks.ModelCheckpoint(
    "best_churn_model.h5",
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# =========================
# TRAIN FINAL MODEL
# =========================
best_params = grid_result.best_params_

final_model = build_model(
    neurons=best_params['model__neurons'],
    dropout_rate=best_params['model__dropout_rate'],
    optimizer=best_params['optimizer']
)

history = final_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=best_params['epochs'],
    batch_size=best_params['batch_size'],
    callbacks=[checkpoint, early_stop],
    verbose=1
)

# =========================
# EVALUATION
# =========================
loss, accuracy = final_model.evaluate(X_test, y_test)
print("\nFinal Test Accuracy:", accuracy)

# =========================
# LOAD BEST MODEL
# =========================
best_model = keras.models.load_model("best_churn_model.h5")
best_loss, best_acc = best_model.evaluate(X_test, y_test)

print("Best Saved Model Accuracy:", best_acc)

/tmp/ipykernel_2139749/1339878628.py:28: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Dataset Loaded
   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63       0 

2026-04-17 14:07:26.949128: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-17 14:07:26.977443: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-17 14:07:27.045996: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-17 14:07:27.045998: I


Best Parameters: {'batch_size': 32, 'epochs': 50, 'model__dropout_rate': 0.2, 'model__neurons': 64, 'optimizer': 'adam'}
Best Cross-Validation Accuracy: 0.8578749999999999
Epoch 1/50


/opt/intel/oneapi/intelpython/lib/python3.11/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8462 - loss: 0.3681 - val_accuracy: 0.8581 - val_loss: 0.3490
Epoch 8/50

Epoch 8: val_accuracy did not improve from 0.85812
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8494 - loss: 0.3652 - val_accuracy: 0.8575 - val_loss: 0.3486
Epoch 9/50

Epoch 9: val_accuracy improved from 0.85812 to 0.85875, saving model to best_churn_model.h5
 - loss: 0.3842 

200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8459 - loss: 0.3697 - val_accuracy: 0.8587 - val_loss: 0.3467
Epoch 10/50

Epoch 10: val_accuracy did not improve from 0.85875
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8450 - loss: 0.3594 - val_accuracy: 0.8562 - val_loss: 0.3478
Epoch 11/50

Epoch 11: val_accuracy did not improve from 0.85875
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8523 - loss: 0.3590 - val_accuracy: 0.8587 - val_loss: 0.3453
Epoch 12/50

Epoch 12: val_accuracy improved from 0.85875 to 0.86000, saving model to best_churn_model.h5
- loss: 0.3472 

200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8489 - loss: 0.3561 - val_accuracy: 0.8600 - val_loss: 0.3459
Epoch 13/50

Epoch 13: val_accuracy did not improve from 0.86000
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8525 - loss: 0.3560 - val_accuracy: 0.8594 - val_loss: 0.3455
Epoch 14/50

Epoch 14: val_accuracy did not improve from 0.86000
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8470 - loss: 0.3542 - val_accuracy: 0.8594 - val_loss: 0.3436
Epoch 15/50

Epoch 15: val_accuracy did not improve from 0.86000
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8614 - loss: 0.3475 - val_accuracy: 0.8600 - val_loss: 0.3447
Epoch 16/50

Epoch 16: val_accuracy did not improve from 0.86000
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8567 - loss: 0.3515 - val_accuracy: 0.8594 - val_loss: 0.3411
Epoch 17/50

Epoch 17: val_accuracy did not improve from 0.86000
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8514 - loss: 0.3537 - val_accuracy:

200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8580 - loss: 0.3476 - val_accuracy: 0.8606 - val_loss: 0.3404
Epoch 22/50

Epoch 22: val_accuracy did not improve from 0.86063
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8589 - loss: 0.3446 - val_accuracy: 0.8587 - val_loss: 0.3437
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8630 - loss: 0.3359 



Final Test Accuracy: 0.8629999756813049
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8595 - loss: 0.3377  
Best Saved Model Accuracy: 0.859499990940094
